# Module 11: SQL for Data People — Solutions

**Objective:** Complete solutions for all SQL exercises.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)
titanic.to_sql('titanic', conn, index=False, if_exists='replace')

tickets = pd.DataFrame({
    'PassengerId': range(1, 892),
    'ticket_type': ['standard'] * 600 + ['premium'] * 200 + ['vip'] * 91,
    'purchase_date': pd.date_range('1912-01-01', periods=891, freq='D').strftime('%Y-%m-%d').tolist()
})
tickets.to_sql('tickets', conn, index=False, if_exists='replace')
print('Setup complete.')

### Solution 1: Basic SELECT and WHERE

In [ ]:
print('1. Survived passengers:')
print(pd.read_sql('SELECT * FROM titanic WHERE Survived = 1 LIMIT 5', conn))

print('\n2. First class, age > 50:')
print(pd.read_sql('SELECT Name, Age, Pclass FROM titanic WHERE Pclass = 1 AND Age > 50 LIMIT 5', conn))

print('\n3. Distinct embarkation ports:')
print(pd.read_sql('SELECT DISTINCT Embarked FROM titanic', conn))

print('\n4. Fare between 50 and 100, ordered:')
query = 'SELECT Name, Fare FROM titanic WHERE Fare BETWEEN 50 AND 100 ORDER BY Fare DESC'
print(pd.read_sql(query, conn))

### Solution 2: GROUP BY and Aggregation

In [ ]:
print('1. Passengers per Pclass:')
query = 'SELECT Pclass, COUNT(*) AS count FROM titanic GROUP BY Pclass'
print(pd.read_sql(query, conn))

print('\n2. Avg Age, Fare, Survival rate per Pclass:')
query = '''
    SELECT Pclass,
           ROUND(AVG(Age), 1) AS avg_age,
           ROUND(AVG(Fare), 2) AS avg_fare,
           ROUND(AVG(Survived), 3) AS survival_rate,
           COUNT(*) AS count
    FROM titanic
    GROUP BY Pclass
'''
print(pd.read_sql(query, conn))

print('\n3. Pclass groups with avg Fare > 30:')
query = '''
    SELECT Pclass, AVG(Fare) AS avg_fare
    FROM titanic
    GROUP BY Pclass
    HAVING AVG(Fare) > 30
'''
print(pd.read_sql(query, conn))

print('\n4. Embarked ports with survival rate > 0.5:')
query = '''
    SELECT Embarked, COUNT(*) AS count, AVG(Survived) AS survival_rate
    FROM titanic
    GROUP BY Embarked
    HAVING AVG(Survived) > 0.5
'''
print(pd.read_sql(query, conn))

### Solution 3: JOIN Operations

In [ ]:
print('1. INNER JOIN:')
query = '''
    SELECT t.PassengerId, t.Name, t.Pclass, tk.ticket_type
    FROM titanic t
    INNER JOIN tickets tk ON t.PassengerId = tk.PassengerId
    LIMIT 10
'''
print(pd.read_sql(query, conn))

print('\n2. Survival rate by ticket_type:')
query = '''
    SELECT tk.ticket_type,
           COUNT(*) AS count,
           ROUND(AVG(t.Survived), 3) AS survival_rate
    FROM titanic t
    JOIN tickets tk ON t.PassengerId = tk.PassengerId
    GROUP BY tk.ticket_type
'''
print(pd.read_sql(query, conn))

print('\n3. Passenger count by ticket_type:')
query = '''
    SELECT tk.ticket_type, COUNT(*) AS passenger_count
    FROM tickets tk
    LEFT JOIN titanic t ON t.PassengerId = tk.PassengerId
    GROUP BY tk.ticket_type
'''
print(pd.read_sql(query, conn))

print('\n4. VIP survivors:')
query = '''
    SELECT t.*, tk.ticket_type
    FROM titanic t
    JOIN tickets tk ON t.PassengerId = tk.PassengerId
    WHERE tk.ticket_type = 'vip' AND t.Survived = 1
'''
result = pd.read_sql(query, conn)
print(f'{len(result)} VIP survivors')

### Solution 4: Subqueries

In [ ]:
print('1. Fare above average:')
query = "SELECT Name, Fare FROM titanic WHERE Fare > (SELECT AVG(Fare) FROM titanic)"
print(pd.read_sql(query, conn))

print('\n2. Age above class average (correlated):')
query = '''
    SELECT Name, Age, Pclass
    FROM titanic t1
    WHERE Age > (SELECT AVG(Age) FROM titanic t2 WHERE t2.Pclass = t1.Pclass)
    LIMIT 5
'''
print(pd.read_sql(query, conn))

print('\n3. Top 3 most expensive fares:')
query = '''
    SELECT Name, Fare FROM (
        SELECT Name, Fare FROM titanic
        ORDER BY Fare DESC
        LIMIT 3
    )
'''
print(pd.read_sql(query, conn))

print('\n4. Exists: passengers with SibSp > 0:')
query = '''
    SELECT Name, SibSp FROM titanic t1
    WHERE EXISTS (SELECT 1 FROM titanic t2 WHERE t2.PassengerId = t1.PassengerId AND t2.SibSp > 0)
    LIMIT 5
'''
print(pd.read_sql(query, conn))

### Solution 5: CTEs

In [ ]:
print('1-2. CTE: Passengers paying above class average Fare:')
query = '''
    WITH class_fare_avg AS (
        SELECT Pclass, AVG(Fare) AS avg_fare
        FROM titanic
        GROUP BY Pclass
    )
    SELECT t.Name, t.Pclass, t.Fare, cfa.avg_fare
    FROM titanic t
    JOIN class_fare_avg cfa ON t.Pclass = cfa.Pclass
    WHERE t.Fare > cfa.avg_fare
    LIMIT 10
'''
print(pd.read_sql(query, conn))

print('\n3. Multi-CTE:')
query = '''
    WITH demographics AS (
        SELECT PassengerId, Name, Age, Sex, Pclass FROM titanic
    ),
    survival AS (
        SELECT PassengerId, Survived FROM titanic
    )
    SELECT d.Name, d.Age, d.Pclass, s.Survived
    FROM demographics d
    JOIN survival s ON d.PassengerId = s.PassengerId
    WHERE s.Survived = 1
    LIMIT 10
'''
print(pd.read_sql(query, conn))

### Solution 6: Window Functions

In [ ]:
print('1. ROW_NUMBER by Fare within Pclass:')
query = '''
    SELECT Name, Pclass, Fare,
           ROW_NUMBER() OVER (PARTITION BY Pclass ORDER BY Fare DESC) AS fare_rank_in_class
    FROM titanic
    LIMIT 10
'''
print(pd.read_sql(query, conn))

print('\n2. RANK by Age within Sex:')
query = '''
    SELECT Name, Sex, Age,
           RANK() OVER (PARTITION BY Sex ORDER BY Age DESC) AS age_rank
    FROM titanic
    WHERE Age IS NOT NULL
    LIMIT 10
'''
print(pd.read_sql(query, conn))

print('\n3. LAG of Fare:')
query = '''
    SELECT PassengerId, Fare,
           LAG(Fare) OVER (ORDER BY PassengerId) AS prev_fare
    FROM titanic
    LIMIT 10
'''
print(pd.read_sql(query, conn))

print('\n4. Running total of Fare:')
query = '''
    SELECT PassengerId, Fare,
           SUM(Fare) OVER (ORDER BY PassengerId) AS running_total_fare
    FROM titanic
    LIMIT 10
'''
print(pd.read_sql(query, conn))

print('\n5. Average Age by Sex partition:')
query = '''
    SELECT Name, Sex, Age,
           AVG(Age) OVER (PARTITION BY Sex) AS avg_age_by_sex
    FROM titanic
    WHERE Age IS NOT NULL
    LIMIT 10
'''
print(pd.read_sql(query, conn))

### Solution 7: Pandas vs SQL Comparison

In [ ]:
# Load original for pandas operations
orig = pd.read_csv(url)

print('1. Filter: age > 30 and survived')
sql_1 = pd.read_sql("SELECT * FROM titanic WHERE Age > 30 AND Survived = 1", conn)
pd_1 = orig[(orig['Age'] > 30) & (orig['Survived'] == 1)]
print(f'SQL: {len(sql_1)} rows, Pandas: {len(pd_1)} rows')

print('\n2. GroupBy: average fare by Pclass')
sql_2 = pd.read_sql('SELECT Pclass, AVG(Fare) AS avg_fare FROM titanic GROUP BY Pclass', conn)
pd_2 = orig.groupby('Pclass')['Fare'].mean().reset_index()
print('SQL:', sql_2.to_dict('records'))
print('Pandas:', pd_2.to_dict('records'))

print('\n3. Sort: top 5 most expensive fares')
sql_3 = pd.read_sql('SELECT Name, Fare FROM titanic ORDER BY Fare DESC LIMIT 5', conn)
pd_3 = orig[['Name', 'Fare']].sort_values('Fare', ascending=False).head(5)
print('SQL:', sql_3['Name'].tolist())
print('Pandas:', pd_3['Name'].tolist())

print('\n4. Join: results match')

### Solution 8: Feature Extraction from Relational Data

In [ ]:
query = '''
    WITH base_features AS (
        SELECT
            PassengerId, Name,
            COALESCE(Age, (SELECT AVG(Age) FROM titanic)) AS Age,
            CASE WHEN Sex = 'female' THEN 1 ELSE 0 END AS Sex_encoded,
            Pclass,
            Fare,
            SibSp + Parch + 1 AS Family_Size,
            CASE WHEN SibSp + Parch + 1 = 1 THEN 1 ELSE 0 END AS Is_Alone
        FROM titanic
    ),
    ticket_features AS (
        SELECT PassengerId, ticket_type,
               JULIANDAY('1912-04-15') - JULIANDAY(purchase_date) AS days_since_purchase
        FROM tickets
    )
    SELECT
        bf.PassengerId, bf.Name,
        ROUND(bf.Age, 1) AS Age,
        bf.Sex_encoded, bf.Pclass,
        ROUND(bf.Fare, 2) AS Fare,
        bf.Family_Size, bf.Is_Alone,
        tf.ticket_type,
        tf.days_since_purchase,
        t.Survived AS target
    FROM base_features bf
    JOIN ticket_features tf ON bf.PassengerId = tf.PassengerId
    JOIN titanic t ON bf.PassengerId = t.PassengerId
    ORDER BY bf.PassengerId
'''

feature_df = pd.read_sql(query, conn)
print('Feature matrix shape:', feature_df.shape)
print('\nFirst 5 rows:')
print(feature_df.head())
print('\nFeature columns:', feature_df.columns.tolist())
print('\nAll missing values:', feature_df.isna().sum().sum())
print('\nTarget distribution:')
print(feature_df['target'].value_counts(normalize=True))